# Notebook 3: Prompt Chaining with LangChain (LCEL)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain what **Prompt Chaining** is and why it outperforms single-prompt approaches.
2. Use the **pipe operator (`|`)** in LangChain Expression Language (LCEL).
3. Build **sequential chains** where each step's output feeds into the next.
4. Use **`RunnablePassthrough`** to carry original inputs alongside computed outputs.
5. Use **`RunnableParallel`** to run independent steps simultaneously.
6. Run a working **Earnings Report Analyzer** that chains sentiment, metrics, and summary steps.

---

## Use Case: Earnings Report Analyzer

> **Scenario**: When a company releases its quarterly earnings, analysts need to quickly extract multiple insights. A single prompt asking "analyze this report" produces mediocre results. Breaking it into focused steps produces professional-grade analysis.

**Our 3-Stage Pipeline:**

```
Earnings Report Text
        │
        ├──────────────────────────────────────────────────┐
        │                                                  │
        ▼                                                  ▼
[Stage 1: Sentiment]                          [Stage 2: Metrics]
"Is this Bullish/Bearish/Neutral?"           "Extract Revenue, EPS, Guidance"
        │                                                  │
        └──────────────────┬───────────────────────────────┘
                           │
                           ▼
                  [Stage 3: Summary]
          "Write a 3-sentence analyst memo
           using the original text + sentiment + metrics"
                           │
                           ▼
               Professional Analyst Summary
```

---

## Key Concepts

| Concept | Syntax | Description |
|---|---|---|
| **Chain** | `prompt \| llm \| parser` | Sequential composition using the pipe operator |
| **StrOutputParser** | `StrOutputParser()` | Extracts the string content from LLM response |
| **RunnablePassthrough** | `RunnablePassthrough()` | Passes input through unchanged |
| **assign()** | `RunnablePassthrough.assign(key=chain)` | Runs a chain and adds result to the input dict |
| **RunnableParallel** | `{"a": chain_a, "b": chain_b}` | Runs multiple chains simultaneously |

## ⚙️ Step 1: Setup and Azure OpenAI Configuration

In [ ]:
# Install required packages (uncomment if needed)
# !pip install langchain-openai langchain

import os
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

# ── Load credentials from .env ─────────────────────────────────────────────────
load_dotenv()

llm = AzureChatOpenAI(azure_deployment="gpt-4.1", temperature=0.2)
print("Azure OpenAI model initialized!")

## 🔗 Step 2: Understanding the Pipe Operator

The pipe operator (`|`) is the core of LCEL. It chains runnables together so that the output of one becomes the input of the next.

```python
# These two are equivalent:
chain = prompt | llm | parser

# vs.

chain = parser.invoke(llm.invoke(prompt.invoke(input)))
```

Let's start with a simple example: a two-step chain that finds a company's ticker, then identifies its sector.

In [ ]:
# Simple sequential chain: Company → Ticker → Sector
simple_chain = (
    ChatPromptTemplate.from_template("What is the stock ticker symbol for {company}? Reply with just the ticker symbol.")
    | llm
    | StrOutputParser()
    | ChatPromptTemplate.from_template("What sector does the company with ticker symbol '{text}' operate in? Be brief (1 sentence).")
    | llm
    | StrOutputParser()
)

print("Testing simple sequential chain...")
result = simple_chain.invoke({"company": "Microsoft"})
print(f"Result: {result}")

## 📄 Step 3: The Sample Earnings Report

Let's define a realistic earnings report that we'll analyze through our pipeline.

In [ ]:
earnings_report = """
TECHCORP INC. — Q3 2025 EARNINGS CALL SUMMARY

TechCorp Inc. (NASDAQ: TCORP) reported record quarterly revenue of $12.5 billion for Q3 2025, 
representing a 15% increase year-over-year and beating analyst consensus estimates of $11.8 billion.

Earnings per share (EPS) came in at $1.45, compared to the expected $1.30, representing a 
11.5% beat. Operating margins remained healthy at 24%, though slightly compressed from 26% 
in the prior year quarter due to increased R&D investment in artificial intelligence initiatives.

Looking ahead, management provided Q4 2025 revenue guidance of $10.5B–$11.0B, below analyst 
expectations of $12.0B, citing supply chain headwinds in the semiconductor space and 
macroeconomic uncertainty in key European markets.

CEO Jane Smith stated: 'While we are proud of our Q3 performance, we are taking a prudent 
approach to Q4 guidance given the current macro environment.'
"""

print("✅ Sample earnings report loaded.")
print(f"   Length: {len(earnings_report)} characters")

## 🏗️ Step 4: Building the Three Analysis Chains

Each chain has a focused, single responsibility.

In [ ]:
# ── Chain 1: Sentiment Analysis ──────────────────────────────────────────────
sentiment_prompt = ChatPromptTemplate.from_template(
    """You are a sell-side equity analyst. Analyze the investor sentiment of this earnings report.

    Earnings Report:
    {report}

    Provide:
    1. Overall Sentiment: (Bullish / Bearish / Mixed)
    2. Key Positive Factors: (1-2 bullet points)
    3. Key Negative Factors: (1-2 bullet points)
    """
)
sentiment_chain = sentiment_prompt | llm | StrOutputParser()

# ── Chain 2: Key Metrics Extraction ──────────────────────────────────────────
metrics_prompt = ChatPromptTemplate.from_template(
    """Extract the following key financial metrics from this earnings report. 
    Format as a clean, structured list.

    Earnings Report:
    {report}

    Extract:
    - Revenue (Actual vs. Estimate)
    - EPS (Actual vs. Estimate)
    - Operating Margin
    - Forward Guidance
    - Beat/Miss Status
    """
)
metrics_chain = metrics_prompt | llm | StrOutputParser()

# ── Chain 3: Investment Summary ───────────────────────────────────────────────
summary_prompt = ChatPromptTemplate.from_template(
    """You are a Senior Equity Analyst writing for institutional investors.
    Write a concise, professional 3-sentence investment summary.

    Original Report:
    {report}

    Sentiment Analysis:
    {sentiment}

    Key Metrics:
    {metrics}

    Investment Summary (3 sentences, professional tone):
    """
)
summary_chain = summary_prompt | llm | StrOutputParser()

print("✅ All three analysis chains defined.")

## 🚀 Step 5: Assembling the Full Pipeline with RunnablePassthrough

`RunnablePassthrough.assign()` is the key to multi-input chains. It:
1. Passes the original input through unchanged.
2. Runs the specified chain and adds its result to the input dictionary under the given key.

```python
# This is what happens step by step:
# Input: "earnings report text"
# After {"report": RunnablePassthrough()}: {"report": "earnings report text"}
# After .assign(sentiment=...): {"report": "...", "sentiment": "Bullish..."}
# After .assign(metrics=...):   {"report": "...", "sentiment": "...", "metrics": "Revenue: $12.5B..."}
# summary_chain receives all three fields and writes the final memo
```

In [ ]:
# Full pipeline: wrap text → add sentiment → add metrics → final summary
full_analysis_chain = (
    {"report": RunnablePassthrough()}          # Step 0: Wrap input as dict
    | RunnablePassthrough.assign(              # Step 1: Add sentiment
        sentiment=sentiment_chain
    )
    | RunnablePassthrough.assign(              # Step 2: Add metrics
        metrics=metrics_chain
    )
    | summary_chain                            # Step 3: Write final summary
)

print("✅ Full analysis pipeline assembled.")
print("   Flow: text → {report} → +sentiment → +metrics → summary")

## 🧪 Step 6: Running the Full Pipeline

In [ ]:
import time

print("📊 Running full earnings analysis pipeline...\n")
start = time.time()

# Run the full chain
final_summary = full_analysis_chain.invoke(earnings_report)

elapsed = time.time() - start
print(f"⏱️ Total time: {elapsed:.1f}s\n")
print("=" * 60)
print("📝 FINAL ANALYST SUMMARY")
print("=" * 60)
print(final_summary)

## 🔀 Step 7: Parallel Execution for Independent Steps

Sentiment and Metrics analysis are **independent** — they both only need the original report. We can run them in parallel to save time!

In [ ]:
# Parallel version: run sentiment and metrics simultaneously
parallel_analysis = RunnableParallel(
    sentiment=sentiment_chain,
    metrics=metrics_chain
)

# Then feed both results into the summary
parallel_full_chain = (
    {"report": RunnablePassthrough()}
    | RunnablePassthrough.assign(
        parallel_results=parallel_analysis
    )
    | (lambda x: {
        "report": x["report"],
        "sentiment": x["parallel_results"]["sentiment"],
        "metrics": x["parallel_results"]["metrics"]
    })
    | summary_chain
)

print("Testing parallel version...")
start = time.time()
parallel_summary = parallel_full_chain.invoke(earnings_report)
elapsed_parallel = time.time() - start

print(f"⏱️ Parallel execution time: {elapsed_parallel:.1f}s")
print("=" * 60)
print("📝 PARALLEL ANALYST SUMMARY")
print("=" * 60)
print(parallel_summary)

## 📚 Summary and Key Takeaways

| Pattern | When to Use | Code |
|---|---|---|
| **Sequential Chain** | Step B needs Step A's output | `chain_a \| chain_b` |
| **RunnablePassthrough** | Pass original input to later steps | `{"key": RunnablePassthrough()}` |
| **assign()** | Add computed fields to the input dict | `RunnablePassthrough.assign(key=chain)` |
| **RunnableParallel** | Independent steps that can run simultaneously | `RunnableParallel(a=chain_a, b=chain_b)` |

### 🔑 Key Insights

1. **Focus beats breadth**: Giving the LLM one focused task per step (sentiment OR metrics, not both) produces higher quality results than a single mega-prompt.
2. **LCEL is composable**: Chains can be nested, combined, and reused like building blocks.
3. **Parallel when independent**: If Step A and Step B don't depend on each other, run them in parallel to halve the latency.

### 🚀 Next Steps
- **Notebook 4**: Learn the Evaluator-Optimiser pattern for self-correcting outputs.
